# 03 Fine-Tuning Gemma 4 2B-E2B

Minimal Phase 2 walkthrough for loading pose/text pairs, attaching LoRA adapters, running one training epoch, plotting loss, and saving a checkpoint.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml
from torch.utils.data import DataLoader

from src.data.pose_to_text_dataset import PoseAugmentationConfig, PoseToTextDataset, collate_pose_text_batch
from src.models.gemma_finetune import FineTuneConfig, train_gemma
from src.models.gemma_loader import load_gemma_4_2b_e2b

config = yaml.safe_load(Path("config.yaml").read_text(encoding="utf-8"))
device = "cuda" if torch.cuda.is_available() else "cpu"
config, device

In [ ]:
train_csv = Path(config["data"]["training_pairs_dir"]) / "train.csv"
val_csv = Path(config["data"]["training_pairs_dir"]) / "val.csv"

augmentation = PoseAugmentationConfig(
    enabled=config["data"]["augmentation"]["enabled"],
    rotation_degrees=float(config["data"]["augmentation"]["rotation_degrees"]),
    speed_variation=float(config["data"]["augmentation"]["speed_variation"]),
)

train_dataset = PoseToTextDataset.from_csv(
    train_csv,
    include_face=bool(config["data"]["include_face"]),
    augmentation=augmentation,
)
val_dataset = PoseToTextDataset.from_csv(
    val_csv,
    include_face=bool(config["data"]["include_face"]),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=int(config["training"]["batch_size"]),
    shuffle=True,
    num_workers=int(config["training"]["num_workers"]),
    collate_fn=collate_pose_text_batch,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=int(config["training"]["eval_batch_size"]),
    shuffle=False,
    num_workers=int(config["training"]["num_workers"]),
    collate_fn=collate_pose_text_batch,
)

len(train_dataset), len(val_dataset)

In [ ]:
model, tokenizer = load_gemma_4_2b_e2b(
    lora_rank=int(config["lora"]["rank"]),
    load_in_4bit=bool(config["gemma"]["load_in_4bit"]),
    max_seq_length=int(config["gemma"]["max_seq_length"]),
)

train_config = FineTuneConfig(
    output_dir=Path(config["gemma"]["checkpoint_dir"]),
    learning_rate=float(config["training"]["learning_rate"]),
    weight_decay=float(config["training"]["weight_decay"]),
    warmup_steps=int(config["training"]["warmup_steps"]),
    num_epochs=1,
    gradient_accumulation_steps=int(config["training"]["gradient_accumulation_steps"]),
    max_grad_norm=float(config["training"]["max_grad_norm"]),
    early_stopping_patience=int(config["training"]["early_stopping_patience"]),
    log_every_steps=int(config["training"]["log_every_steps"]),
    checkpoint_every_epochs=1,
    metric_for_best_model=str(config["training"]["metric_for_best_model"]),
    maximize_metric=bool(config["training"]["maximize_metric"]),
    device=device,
)
train_config

In [ ]:
history = train_gemma(
    model=model,
    tokenizer=tokenizer,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    config=train_config,
)
history

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.train_loss, label="train loss")
plt.plot(history.val_loss, label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Gemma LoRA Fine-Tuning")
plt.legend()
plt.show()

In [ ]:
print("Best checkpoint:", history.best_checkpoint)
print("Best metric:", history.best_metric)